# Daily H=22, notebook 1/3: dados e benchmarks estatísticos

Este notebook cria o target Yang–Zhang diário corrigido e executa RW, HAR,
GARCH(1,1), VIX raw, VIX calibrado e XGBoost em janelas 8/2/1. Ele não altera
os resultados da execução v1.

**Tempo estimado no Colab:** 10–20 minutos. O teto operacional é 60 minutos,
deixando ao menos 10 minutos para empacotar e baixar os resultados antes do
limite de 70 minutos.

In [ ]:
# Configuração compartilhada: repositório + armazenamento persistente
from pathlib import Path
import os, shutil, subprocess, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('(Not on Colab / Drive mount skipped):', exc)

REPO_URL = 'https://github.com/hugogobato/Mestrado_Anna_Julia.git'
REPO = Path('/content/Mestrado_Anna_Julia')
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

EXP = REPO / 'experiments/daily_h22'
PERSIST = Path('/content/drive/MyDrive/Mestrado_Anna_Julia/daily_h22')
DATA_DIR = PERSIST / 'data'
RESULTS_DIR = PERSIST / 'results'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ['DAILY_H22_DATA_DIR'] = str(DATA_DIR)
os.environ['DAILY_H22_RESULTS_DIR'] = str(RESULTS_DIR)
os.environ['MPLCONFIGDIR'] = '/content/matplotlib_cache'
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(EXP / 'requirements-colab.txt')], check=True)

def run_script(name, *args):
    command = [sys.executable, str(EXP / 'src' / name), *map(str, args)]
    print('RUN:', ' '.join(command))
    subprocess.run(command, check=True, env=os.environ.copy())

print('Experiment:', EXP)
print('Persistent data:', DATA_DIR)
print('Persistent results:', RESULTS_DIR)
try:
    import torch
    print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
          '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
except Exception as exc:
    print('Torch check failed:', exc)


## Configuração da sessão

Use `smoke` para testar uma janela rapidamente. Use `final` para produzir os
resultados completos nas 16 janelas em uma única execução.

In [ ]:
RUN_MODE = 'final'  # 'smoke' ou 'final'
SESSION_BUDGET_MINUTES = 60
if RUN_MODE not in {'smoke', 'final'}:
    raise ValueError('RUN_MODE deve ser smoke ou final')
print('Mode:', RUN_MODE, '| session budget:', SESSION_BUDGET_MINUTES, 'minutes')


## Preparação diária

O script gera uma linha por pregão, aplica um atraso conservador às variáveis
macro sem data de publicação e salva o manifesto das 16 janelas 8/2/1.

In [ ]:
prep_args = ['--smoke'] if RUN_MODE == 'smoke' else []
run_script('10_data_prep_daily.py', *prep_args)

import pandas as pd
display(pd.read_csv(DATA_DIR / 'split_manifest.csv'))
panel = pd.read_parquet(DATA_DIR / 'daily_panel.parquet')
display(panel[['date', 'rv_yz_22', 'close_variance_forward_22', 'vix']].tail())


## Benchmarks estatísticos

Cada arquivo parcial é salvo antes de avançar. XGBoost usa 22 regressores
diretos, um por horizonte. O VIX raw é headline apenas em `h=22`.

In [ ]:
args = ['--time-budget-minutes', SESSION_BUDGET_MINUTES]
if RUN_MODE == 'smoke':
    args += ['--smoke']
run_script('11_statistical_benchmarks.py', *args)

partial = sorted((RESULTS_DIR / 'forecasts/partial').glob('*.parquet'))
print('Partial forecast files:', len(partial))
coverage = []
for path in sorted((RESULTS_DIR / 'forecasts').glob('*.parquet')):
    if path.name == 'all_models.parquet':
        continue
    frame = pd.read_parquet(path)
    coverage.append({'file': path.name, 'windows': frame.window_id.nunique(),
                     'rows': len(frame)})
display(pd.DataFrame(coverage))
if RUN_MODE == 'final' and coverage and not all(row['windows'] == 16 for row in coverage):
    raise RuntimeError('Cobertura incompleta: o teto de 60 minutos foi atingido inesperadamente.')


## Diagnóstico rápido

Esta avaliação parcial é útil para detectar problemas de escala. A avaliação
oficial completa será executada no notebook 3 depois dos modelos neurais.

In [ ]:
run_script('15_evaluate_daily.py', '--smoke')
metrics_h22 = RESULTS_DIR / 'metrics/metrics_h22.csv'
if metrics_h22.exists():
    display(pd.read_csv(metrics_h22).sort_values('MSE'))


In [ ]:
ARCHIVE_NAME = 'daily_h22_notebook1_state'
# Empacotamento e download seguro do estado atual
import shutil
from pathlib import Path

archive_base = Path('/content') / ARCHIVE_NAME
output_file = shutil.make_archive(str(archive_base), 'zip', root_dir=PERSIST)
print('Archive created:', output_file)
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
